In [46]:
import cv2 as cv

import numpy as np
import os
import copy

In [47]:
def show_image(image_,
               window_name='image',
               timeout=0):
    """
    Show image.
    :param image_
    :param window_name
    :param timeout
    """

    # reshape image scale to be showed properly
    height, width = image_.shape[:2]

    fx = 0.3
    fy = 0.3
    
    new_width = int(width * fx)
    new_height = int(height * fy)
    
    image_ = cv.resize(image_,(new_width, new_height))
    cv.imshow(window_name, np.uint8(image_))
    cv.waitKey(timeout)
    cv.destroyAllWindows()

In [48]:
#partial template 
template = cv.imread("D:/MASTER/Sem_II/Computer Vision/CV-2023-Project1/board_fixed.jpg")
image = cv.imread("D:/MASTER/Sem_II/Computer Vision/CV-2023-Project1/train/regular_tasks/3_16.jpg")

In [35]:

def get_board(img, src, dst):
    h, w = img.shape[:2]

    M = cv.getPerspectiveTransform(src, dst)

    warped = cv.warpPerspective(img, M, (w, h), flags=cv.INTER_LINEAR)

    return  cv.flip(warped, 1)


w, h = template.shape[0], template.shape[1]



src = np.float32([(20,14),     
                  (1885,17),    
                  (10,1920),    
                  (1892,1886)]) 


dst = np.float32([(w, 0),  
                  (0, 0), 
                  (w, h),   
                  (0, h)])

board_fixed = get_board(template, src, dst)
curr_dir = os.getcwd()
cv.imwrite(os.path.join(curr_dir, 'board_fixed.jpg'), board_fixed)
show_image(board_fixed)
cv.waitKey(0)
cv.destroyAllWindows()

In [40]:
def get_keypoints_and_features(image, show_details=False, max_features=1500) -> tuple:
    """
    :return the keypoints: [cv.Keypoint] and the features: np.ndarray for each keypoint.
    """

    def show_keypoints(image_, keypoints_):
        """
        Show the keypoints found in the image.
        """
        image_output = image_.copy()
        image_output = cv.drawKeypoints(image_, keypoints_, image_output, flags=cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

        show_image(image_output, 'keypoints')

    gray_image = cv.cvtColor(image, cv.COLOR_BGR2GRAY) 
    
    sift = cv.SIFT_create(max_features) 

    keypoints = sift.detect(gray_image, None)

    
    keypoints, features = sift.compute(gray_image, keypoints)
     
    if show_details:
        show_keypoints(image, keypoints)

    return keypoints, features

In [36]:
def match_features(features_source, features_dest) -> [[cv.DMatch]]:
    """
    Match features from the source image with the features from the destination image.
    :return: [[DMatch]] - The rusult of the matching. For each set of features from the source image,
    it returns the first 'K' matchings from the destination images.
    """
    feature_matcher = cv.DescriptorMatcher_create("FlannBased")
    matches = feature_matcher.knnMatch(features_source, features_dest, k=2)

    return matches

In [37]:
def generate_homography(all_matches:  [cv.DMatch],
                        keypoints_source: [cv.KeyPoint],
                        keypoints_dest : [cv.KeyPoint],
                        ratio: float = 0.75,
                        ransac_rep: int = 4.0):
    """
    :param all_matches [DMatch]
    :param keypoints_source [cv.Point]
    :param ratio - Lowe's ratio test (the ratio 1st neighbour distance / 2nd neighbour distance)
    :param keypoints_source: nd.array [Nx2] (x, y coordinates)
    :param keypoints_dest: nd.array [Nx2] (x, y coordinates)
    :param ransac_rep: float. The threshold in the RANSAC algorithm.
    :return: The homography matrix.
    
    class DMatch:
        distance - Distance between descriptors. The lower, the better it is.
        imgIdx - Index of the train image
        queryIdx - Index of the descriptor in query descriptors
        trainIdx - Index of the descriptor in train descriptors
    
    class KeyPoint:
        pt - The x, y coordinates of a point.
    
    """
    if not all_matches:
        return None
    
    matches = [] 
    for match in all_matches:  
        if len(match) == 2 and (match[0].distance / match[1].distance) < ratio:
            matches.append(match[0])
     
    points_source = np.float32([keypoints_source[m.queryIdx].pt for m in matches])

    points_dest = np.float32([keypoints_dest[m.trainIdx].pt for m in matches])

    if len(points_source) > 4:
        homography, status = cv.findHomography(points_source, points_dest, cv.RANSAC, ransac_rep)
        return homography
    else:
        return None

In [38]:
def stitch_images(image, template, show_orig_matches: bool=True):
    """

    Args:
        image (Path): Path to the image to be alligned.
        template (Path): Path to the template
        ratio (float): Lowe's ratio test (the ratio 1st neighbour distance / 2nd neighbour distance)

    Returns:
        
    """
    def show_matches(all_matches_, n=10):
        matches = sorted(all_matches_, key = lambda x:x[0].distance)
        matches = matches[:n] 
        image_output = cv.drawMatchesKnn(template, kpsT,
                                         image, kpsIm,   
                                         matches, None, flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
        show_image(image_output, 'matches')

    kpsT, featT = get_keypoints_and_features(template,show_details=True)
    kpsIm, featIm = get_keypoints_and_features(image,show_details=True)
    print(f"Keypoints lenght: {len(kpsT)},/n Features lenght: {len(featT)}")
    print(f"Keypoints lenght: {len(kpsIm)},/n Features lenght: {len(featIm)}")

    all_matches = match_features(featIm, featT)
    print(f"Total number of matches: {len(all_matches)}")

    if show_orig_matches:
        show_matches(copy.copy(all_matches),n=10)

    H = generate_homography(all_matches, kpsIm, kpsT, 0.6, 4.0)

    (h, w) = template.shape[:2]
    result = cv.warpPerspective(image, H, (w, h))
    
    return result



In [49]:

aligned = stitch_images(image, template, show_orig_matches=True)
show_image(aligned)

Keypoints lenght: 1500,/n Features lenght: 1500
Keypoints lenght: 1501,/n Features lenght: 1501
Total number of matches: 1501


In [12]:
class Line:
    """
    Store a line based on the two points.
    """
    def __init__(self, point_1, point_2):
        self.point_1 = point_1
        self.point_2 = point_2
        

class Point:
    def __init__(self, x, y):
        self.x = np.int32(np.round(x))
        self.y = np.int32(np.round(y))
    
    def get_point_as_tuple(self):
        return (self.x, self.y)
    
        
class Patch:
    """
    Store information about each item (where the mark should be found) in the table. 
    """
    def __init__(self, image_patch, x_min, y_min, x_max, y_max, line_idx, column_idx):
        self.image_patch = image_patch
        self.x_min = x_min
        self.y_min = y_min
        self.x_max = x_max
        self.y_max = y_max
        self.line_idx = line_idx
        self.column_idx = column_idx
        self.has_x: int = 0
    
    def set_x(self, has_x: int):
        assert has_x == 0 or has_x == 1 # convention 
        self.has_x = has_x

In [13]:
def get_patches(lines: [Line], columns: [Line], image, show_patches: bool = False) -> [Patch]:
    """
    It cuts out each box from the table defined by the lines and columns.
    :param lines. The lines that difine the table.
    :param columns. The columns that difine the table.
    :param image. The image containing the table.
    :param show_patches. Determine if the patches will be drawn on the image or not.
    :return : A list with all boxes in the table.
    """
    
    def crop_patch(image_, x_min, y_min, x_max, y_max):
        """
        Crops the bounding box represented by the coordinates.
        """
        return image_[y_min: y_max, x_min: x_max].copy()
    
    def draw_patch(image_, patch: Patch, color: tuple = (255, 0, 255)):
        """
        Draw the bounding box corresponding to the patch on the image.
        """
        cv.rectangle(image_, (patch.x_min, patch.y_min), (patch.x_max, patch.y_max), color=color, thickness=5)
    
    assert image.ndim == 2
    if show_patches: 
        image_color = np.dstack((image, image, image))
  
    lines.sort(key=lambda line: line.point_1.y)
    columns.sort(key=lambda column: column.point_1.x)
    patches = []
    step = 3
    for line_idx in range(len(lines) - 1):
        for col_idx in range(len(columns) - 1):
            current_line = lines[line_idx]
            next_line = lines[line_idx + 1] 
            
            y_min = current_line.point_1.y + step
            y_max = next_line.point_1.y - step
            
            current_col = columns[col_idx]
            next_col = columns[col_idx + 1]
            x_min = current_col.point_1.x + step 
            x_max = next_col.point_1.x - step
            
            patch = Patch(image_patch=crop_patch(image,  x_min, y_min, x_max, y_max),
                          x_min=x_min, y_min=y_min, x_max=x_max, y_max=y_max,
                          line_idx=line_idx, column_idx=col_idx)
            
            if show_patches:
                draw_patch(image_color, patch)
            
            patches.append(patch)
            
    if show_patches:
        show_image(image_color, window_name='patches', timeout=0)
    return patches

In [14]:
def find_patches_template(image, template, show_intermediate_results: bool = False):
    """
    This function finds the 'X' in the entire image using the following steps: 
    1. transforming the query image and the template to grayscale
    2. getting the edges for both images using Canny edge detector with threshold1=100 and threshold2=150
    3. obtaining the displacement vector between the query image and the template image
    4. allign the query image to the template
    5. obtain the first left/right corner of the first table and the second table
    6. getting the vertical/horizontal lines based on the table dimension and the number of vertical/horizontal lines based
    7. cut out the paches (the boxes) from the two tables based on the horizontal and vertical lines 
    """
    image_gray=cv.cvtColor(image,cv.COLOR_BGR2GRAY)
    template_gray=cv.cvtColor(template,cv.COLOR_BGR2GRAY)
    query_image_edges = cv.Canny(image_gray,100,150)
    template_image_edges =  cv.Canny(template_gray,100,150)
    
    if show_intermediate_results: 
        show_image(query_image_edges, window_name='query_image_edges', timeout=0)
        show_image(template_image_edges, window_name='template_image_edges', timeout=0)
        
    

    alligned_image = stitch_images(image, template)
    
    left_corner_1 = Point(x=29, y=14)
    right_corner_1 = Point(x=1905, y=1893) 

    step_x =(right_corner_1.x-left_corner_1.x)/15
    step_y =(right_corner_1.y-left_corner_1.y)/15
     
    
    def get_vertical_and_horizontal_lines(left_corner: Point, step_x, step_y_):
        vertical_lines = []
        horizontal_lines = []
        
        # TODO: compute the vertical/horizontal lines based left corner step and window_size 
        for i in range(0,16):
            x_min=np.uint64(left_corner.x+i*step_x)
            line = Line(Point(x=x_min, y=0), Point(x=x_min, y=5000))
            vertical_lines.append(line)
        for i in range(0,16):
            y_min=np.uint64(left_corner.y+i*step_y_)
            line = Line(Point(x=0, y=y_min), Point(x=5000, y=y_min))
            horizontal_lines.append(line)   
        
        return vertical_lines, horizontal_lines
     

    vertical_lines, horizontal_lines = get_vertical_and_horizontal_lines(left_corner_1, step_x, step_y)
    
    alligned_image_gray = cv.cvtColor(alligned_image,cv.COLOR_BGR2GRAY)
    patches = get_patches(horizontal_lines, vertical_lines, alligned_image_gray, 
                               show_patches=show_intermediate_results)

    
    return patches 

In [15]:
patches = find_patches_template(aligned, template,show_intermediate_results= True)
print(len(patches))

Keypoints lenght: 1501,/n Features lenght: 1501
Keypoints lenght: 1500,/n Features lenght: 1500
Total number of matches: 1500
225


In [16]:
print(len(patches))

225


In [17]:
#get the image for pieces
print(os.path.join(os.getcwd(),"board+dominoes","10.jpg"))
pieces_template = cv.imread(os.path.join(os.getcwd(),"board+dominoes","10.jpg"))
pieces_template = stitch_images(pieces_template, template)
show_image(pieces_template)

d:\MASTER\Sem_II\Computer Vision\CV-2023-Project1\board+dominoes\10.jpg
Keypoints lenght: 1501,/n Features lenght: 1501
Keypoints lenght: 1500,/n Features lenght: 1500
Total number of matches: 1500


In [42]:
#get the patches that have piece, to have all possible pieces
patches_with_pieces = find_patches_template(pieces_template, template,show_intermediate_results= True)
pieces = []
for index in range(0,13,2):
    pieces.append(patches_with_pieces[index].image_patch)
    show_image(patches_with_pieces[index].image_patch)

Keypoints lenght: 1501,/n Features lenght: 1501
Keypoints lenght: 1501,/n Features lenght: 1501
Total number of matches: 1501


In [43]:
for index in range(0,13,2):
    show_image(patches_with_pieces[index].image_patch)

In [44]:
#read all images from first game
curr_dir = os.getcwd()
path = "D:/MASTER/Sem_II/Computer Vision/CV-2023-Project1/train/regular_tasks"
dir_path = os.listdir(path)
files_template = "1_"

file_list = [f for f in dir_path if f.startswith(files_template) and f.endswith(".jpg")]

first_match = []

In [24]:
#get all images from the first game
for file in file_list:
    img = cv.imread(os.path.join(path, file))
    show_image(img)
    first_match.append(img)

In [ ]:
stitched = stitch_images(first_match[0], template, show_orig_matches=True)

Keypoints lenght: 1501,/n Features lenght: 1501
Keypoints lenght: 1500,/n Features lenght: 1500
Total number of matches: 1500


In [25]:
def get_moves_of_game_progress(match):
    
    all_moves_range = (7,7)
    
    template_patches = find_patches_template(template, template)
    for move in match:
        patches = find_patches_template(move, template)
        new_pieces_range = [p for p in patches if (p.line_idx >= all_moves_range[0]-1 and
                                             p.line_idx <= all_moves_range[0]+1 and
                                             p.column_idx >= all_moves_range[1]-1 and
                                             p.column_idx <= all_moves_range[1]+1)]
        
        for field in new_pieces_range:
            template_field = [p for p in template_patches if (p.line_idx == field.line_idx and
                                             p.column_idx == field.line_idx)]
            min_dist = np.Inf
            for piece in patches_with_pieces:
                diff = np.mean(np.square(field.image_patch - piece.image_patch))
                if(diff < min_dist):
                    show_image(piece.image_patch)
                    min_dist = diff
            
        break
        
    for p in new_pieces_range:
        show_image(p.image_patch)
        
get_moves_of_game_progress(first_match)

Keypoints lenght: 1501,/n Features lenght: 1501
Keypoints lenght: 1501,/n Features lenght: 1501
Total number of matches: 1501
Keypoints lenght: 1501,/n Features lenght: 1501
Keypoints lenght: 1500,/n Features lenght: 1500
Total number of matches: 1500


ValueError: operands could not be broadcast together with shapes (119,119) (119,120) 